# 10-bob. 10-BOB. ANOMALIYALARNI ANIQLASH
## Kengaytirilgan amaliy ish: mavsumiylikni hisobga oluvchi detektor

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/10-bob/10-2_mavsumiylikni-hisobga-oluvchi-detektor.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 10-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur sakkiz hafta davomida VPN orqali har soatdagi kirishlar sonini modellashtiradi: ish vaqtida o'rtacha 120 ta, ish kunlarining kechasi 15 ta, dam olish kunlari 6 ta. Oxirgi haftaga uchta hodisa qo'shilgan: seshanba kechasi soat 03:00 da 45 ta kirish (o'g'irlangan hisob ma'lumotlari), shanba kuni 70 ta kirish va chorshanba kuni ish vaqtida 260 ta kirish (parol tanlash). Uch detektor taqqoslanadi; har birining chegarasi shunday tanlanadiki, oxirgi haftada ko'pi bilan 3 ta noto'g'ri signal bo'lsin.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np

rng = np.random.default_rng(10)

In [2]:
# 1. Sun'iy ma'lumot: 8 hafta davomida VPN orqali har soatdagi kirishlar soni
soatlar = np.arange(8 * 7 * 24)
kun_soati, hafta_kuni = soatlar % 24, (soatlar // 24) % 7
ish_vaqti = (kun_soati >= 9) & (kun_soati < 18) & (hafta_kuni < 5)
orta = np.where(ish_vaqti, 120, np.where(hafta_kuni < 5, 15, 6))       # ish kuni, kechasi, dam olish
qiymat = rng.poisson(orta).astype(float)

In [3]:
# 2. Uchta sun'iy hodisa (oxirgi haftada)
h0 = 7 * 7 * 24
hodisalar = {"seshanba 03:00, 45 ta kirish (o'g'irlangan hisob)": h0 + 1 * 24 + 3,
             "shanba 14:00, 70 ta kirish (dam olish kuni)": h0 + 5 * 24 + 14,
             "chorshanba 11:00, 260 ta kirish (parol tanlash)": h0 + 2 * 24 + 11}
for nom, i in hodisalar.items():
    qiymat[i] = int(nom.split(", ")[1].split()[0])

oqitish, sinov = slice(0, h0), slice(h0, None)

In [4]:
# 3. Uch xil detektor; chegara: oxirgi haftadagi noto'g'ri signallar soni 3 tadan oshmasin
def global_chegara(x):
    m, s = x[oqitish].mean(), x[oqitish].std()
    return np.abs(x - m) / s

def mavsumiy(x):
    """Haftaning har bir soati (7x24=168) uchun alohida mediana va MAD: barqaror z-ko'rsatkich."""
    ball = np.zeros_like(x)
    for k in range(168):
        tarix = x[oqitish][k::168]
        med = np.median(tarix)
        mad = 1.4826 * np.median(np.abs(tarix - med)) + 1
        ball[k::168] = np.abs(x[k::168] - med) / mad
    return ball

def ewma(x, alfa=0.3):
    """Eksponensial silliqlangan o'rtacha: keyingi qiymat bashorati va undan og'ish."""
    bashorat, ball, dispersiya = x[0], np.zeros_like(x), 25.0
    for i, v in enumerate(x):
        ball[i] = abs(v - bashorat) / np.sqrt(dispersiya)
        dispersiya = (1 - alfa) * dispersiya + alfa * (v - bashorat) ** 2
        bashorat = (1 - alfa) * bashorat + alfa * v
    return ball

hodisa_soatlari = set(hodisalar.values())
print(f"{'Detektor':<24}{'chegara':>9}  aniqlangan hodisalar")
for nom, f in [("Global z-ko'rsatkich", global_chegara), ("EWMA", ewma), ("Mavsumiy (hafta soati)", mavsumiy)]:
    b = f(qiymat)
    oddiy_ball = np.sort([b[i] for i in range(h0, len(b)) if i not in hodisa_soatlari])[::-1]
    chegara = oddiy_ball[3]                         # 3 ta noto'g'ri signalga ruxsat
    topildi = [n.split(",")[0] for n, i in hodisalar.items() if b[i] > chegara]
    print(f"{nom:<24}{chegara:>9.1f}  {', '.join(topildi) if topildi else '-'}")

Detektor                  chegara  aniqlangan hodisalar
Global z-ko'rsatkich          2.0  chorshanba 11:00
EWMA                         11.5  shanba 14:00
Mavsumiy (hafta soati)        3.6  seshanba 03:00, shanba 14:00, chorshanba 11:00


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Detektor                  chegara  aniqlangan hodisalar
Global z-ko'rsatkich          2.0  chorshanba 11:00
EWMA                         11.5  shanba 14:00
Mavsumiy (hafta soati)        3.6  seshanba 03:00, shanba 14:00, chorshanba 11:00
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 10-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.